# 03 · 대화 이력 압축 — 곱셈을 덧셈으로

01번은 이 표로 끝났습니다.

| 턴 | 보낸 문자수 | input | 누적 input |
|---|---|---|---|
| 1 | 130 | 95 | 95 |
| 2 | 20 | 156 | 251 |
| 3 | 20 | 198 | 449 |
| 4 | 19 | 224 | **673** |

보내는 질문 길이는 그대로인데 **과금 입력만 계속 자랍니다.**
해법을 다루지 않고 끝냈는데, 이번에 그 숙제를 받습니다.

### 02번과 무엇이 다를까요

| | 02번 | **03번** |
|---|---|---|
| 대상 | 단발성 구조 데이터 (JSON 배열) | **누적되는 대화 이력** |
| 압축 성격 | 형식 변환 (무손실) | **오래된 정보 버리기 (손실)** |
| 실패 양상 | 필드를 버려서 답하지 못함 | **시간이 지나서 잊음** |
| 압축 비용 | 0 (문자열 처리) | **요약 전략은 LLM 호출이 듭니다** |

### 비교할 전략 4가지

| 전략 | 방식 | 압축 비용 |
|---|---|---|
| **A 전체 유지** | 아무것도 하지 않습니다 (기준선) | 0 |
| **B 슬라이딩 윈도우** | 최근 N턴만 남깁니다 | 0 |
| **C 요약으로 대체** | 오래된 턴을 자연어 요약으로 바꿉니다 | LLM 호출 |
| **D 핵심만 메모** | 결정과 수치만 구조화해 유지합니다 | LLM 호출 |

### 실험 설계에서 고정한 것

**대화 상대는 모델이 아니라 고정 대본입니다.**
매번 모델이 답하게 하면 전략마다 대화 내용이 달라져 비교가 성립하지 않습니다.
12턴 대본을 고정하고 **전략만 바꿔 같은 대화를 재생**합니다.

## 0. 이 노트북이 하는 일

**한 문장으로** — 12턴짜리 상담 대화를 네 가지 방법으로 줄여보고,
**얼마나 싸지는지**와 **대신 무엇을 잊는지**를 잽니다.

---

### 문제부터

챗봇은 매 턴 **지금까지의 대화 전체**를 다시 보냅니다. 모델은 기억을 못 하기 때문입니다.

```
1턴에 보내는 것 : [1턴]
2턴에 보내는 것 : [1턴][2턴]
3턴에 보내는 것 : [1턴][2턴][3턴]
...
12턴에 보내는 것: [1턴][2턴]...[12턴]     ← 매번 처음부터 다시
```

질문은 한 줄인데 **보내는 양은 계속 커집니다.** 그래서 대화가 길어질수록 비용이 눈덩이가 됩니다.

### 그래서 오래된 대화를 줄여봅니다

12턴째에 각 방법이 모델에게 **실제로 보내는 것**입니다.

```
A 전체 유지     [1][2][3][4][5][6][7][8][9][10][11][12]   ← 안 줄임 (기준선)
B 슬라이딩                                  [10][11][12]   ← 최근 3턴만
C 요약으로 대체  (앞부분을 줄글 요약으로)      [10][11][12]
D 핵심만 메모    (앞부분을 키:값 목록으로)     [10][11][12]
```

- **B** 는 공짜로 확 줄지만 앞부분을 **통째로 버립니다**
- **C·D** 는 앞부분을 압축해서 남깁니다. 대신 **요약하려고 LLM 을 한 번 더 부릅니다**

### 순서대로 확인할 것

1. **얼마나 줄었나** — 턴마다 토큰을 실제로 재봅니다
2. **요약이 공짜인가** — C·D 가 쓴 LLM 비용을 더해서 다시 계산합니다
3. **C 와 D 는 뭐가 다른가** — 요약된 결과물을 눈으로 읽어봅니다
4. **줄인 대가는 뭔가** — 앞부분에서 정한 것들(요금제·이메일·위약금)을 기억하는지 물어봅니다
5. **어떻게 틀렸나** — 틀린 답을 그대로 읽어봅니다. "모름"과 "틀린 값을 자신 있게 말함"은 위험도가 다릅니다
6. **그래서 뭘 쓰나** — 비용과 정확도를 같이 놓고 고릅니다
7. **반전: 돈은 정말 줄었나** — 캐시까지 넣고 재보면 순위가 바뀔 수 있습니다

> **마지막 7번이 핵심입니다.**
> 프롬프트 캐시는 앞부분이 **1,024토큰 이상이고 매번 똑같을 때** 걸립니다.
> 그런데 압축은 컨텍스트를 **짧게** 만들고(→ 문턱 미달) 앞부분을 **바꿉니다**(→ 불일치).
> 실제로 1~6번에서는 네 방법 모두 캐시가 한 번도 걸리지 않습니다.
> 캐시가 걸리는 상황을 따로 만들어 보면 **"토큰을 줄였는데 돈은 더 나오는"** 경우가 나옵니다.

## 준비 1 · 설정

**질문** — 실험 환경을 어떻게 갖출까요

- 엔드포인트와 배포명은 `.env` 에서 읽습니다 (01·02번과 동일합니다)
- `measure()` — 텍스트가 몇 input 토큰인지 **API 로 실측**합니다. 추정하지 않습니다
- `generate()` — 모델 응답과 `Usage` 를 함께 돌려줍니다.
  요약 전략의 **압축기 비용**을 추적해야 하기 때문입니다

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 실험 환경을 갖춥니다.
#   · measure()  — 텍스트의 input 토큰을 API 로 실측 (추정하지 않습니다)
#   · generate() — 모델 응답 + Usage. 요약 전략의 '압축기 비용' 추적용
# API 호출: 0회 (준비만)
# ──────────────────────────────────────────────────────────────────────────
import json, os, subprocess, sys, shutil, time, uuid
import urllib.request, urllib.error
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

from nbtools import Usage, Price, show_table

load_dotenv(find_dotenv(usecwd=True) or str(Path.cwd() / ".env"), override=False)


def require(name):
    v = os.environ.get(name)
    if not v:
        raise RuntimeError(f"{name} 가 없습니다. `cp .env.example .env` 후 값을 채우세요.")
    return v


ENDPOINT   = require("AZURE_OPENAI_ENDPOINT").rstrip("/")
DEPLOYMENT = require("AZURE_OPENAI_DEPLOYMENT")

AZ_CANDIDATES = [os.environ.get("AZ_CLI"), shutil.which("az"),
                 "/opt/homebrew/bin/az", "/usr/local/bin/az",
                 str(Path.home() / ".local/bin/az")]


def find_az():
    for c in AZ_CANDIDATES:
        if c and Path(c).exists():
            return c
    raise RuntimeError("az CLI 를 찾지 못했습니다. .env 에 AZ_CLI=/전체/경로/az 를 넣으세요.")


def auth_headers():
    key = os.environ.get("AZURE_OPENAI_API_KEY")
    if key:
        return {"api-key": key}
    r = subprocess.run([find_az(), "account", "get-access-token",
                        "--scope", "https://cognitiveservices.azure.com/.default", "-o", "json"],
                       capture_output=True, text=True, timeout=90)
    if r.returncode != 0:
        raise RuntimeError(f"az 토큰 발급 실패 (`az login` 필요?)\n{r.stderr.strip()[:300]}")
    return {"Authorization": "Bearer " + json.loads(r.stdout)["accessToken"]}


HEADERS = auth_headers()


def responses(input_text, **params):
    url = f"{ENDPOINT}/openai/v1/responses?api-version=preview"
    req = urllib.request.Request(
        url, data=json.dumps({"model": DEPLOYMENT, "input": input_text, **params}).encode(),
        headers={"Content-Type": "application/json", **HEADERS}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            return json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8','replace')[:400]}")


def rtext(resp):
    out = []
    for item in resp.get("output", []):
        for c in item.get("content", []):
            if c.get("type") in ("output_text", "text"):
                out.append(c.get("text", ""))
    return "".join(out).strip()


def measure(text):
    """이 텍스트가 실제로 몇 input 토큰인지 API 로 실측."""
    r = responses(text, max_output_tokens=16)
    return Usage.from_response(r, model=DEPLOYMENT).input_tokens


def generate(prompt, max_tokens=300):
    """(응답텍스트, Usage) — 압축기 비용을 추적하려고 Usage 도 돌려준다."""
    r = responses(prompt, max_output_tokens=max_tokens, temperature=0)
    return rtext(r), Usage.from_response(r, model=DEPLOYMENT)


print("배포:", DEPLOYMENT, "· 준비 완료")

## 준비 2 · 대본 만들기

**질문** — 무엇으로 실험할까요

**왜 대본을 고정할까요** — 매번 모델이 답하게 하면 전략마다 대화 내용이 달라집니다.
그러면 토큰 차이가 *전략 때문인지 대화 때문인지* 알 수 없습니다.
대본을 고정해야 **전략만이 유일한 변수**가 됩니다.

**심어둔 사실** — 4번에서 이것들을 기억하는지 물어봅니다.

| 턴 | 심어둔 것 | 왜 이걸 골랐을까요 |
|---|---|---|
| 3 | 요금제 **번복** — 라이트 고려 → 프리미엄 확정 | 요약이 "라이트"를 남기면 오답입니다 |
| 6 | 청구 이메일 `kim.mj@example.com` | 식별자가 살아남는지 봅니다 |
| 9 | 위약금 **32,000원** | 숫자가 절단되는지 봅니다 |
| 11 | 결합 회선 **3개** | 최근 정보라 **모든 전략이 맞혀야 정상**입니다 |

앞쪽(3·6턴)은 오래돼서 압축 대상이 되고, 뒤쪽(11턴)은 최근이라 안전합니다.
**앞쪽을 얼마나 지켜내는가**가 전략의 실력입니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 전략만이 유일한 변수가 되도록 대화를 고정합니다.
#   · 모델이 매번 답하면 전략마다 대화가 달라져 비교가 성립하지 않습니다
#   · 3·6·9·11턴에 나중에 물어볼 사실을 심어 두었습니다
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
TRANSCRIPT = [
    ("요금제를 바꾸고 싶은데요.",
     "네, 어떤 요금제로 변경을 원하시나요? 현재 5G 라이트 33,000원 요금제를 쓰고 계십니다."),
    ("종류가 어떻게 되나요?",
     "5G 라이트 33,000원, 5G 스탠다드 55,000원, 5G 프리미엄 89,000원 세 가지가 있습니다."),
    ("음... 라이트로 갈까 하다가, 그냥 프리미엄으로 할게요.",
     "네, 5G 프리미엄 89,000원으로 확정하겠습니다. 데이터 무제한이 제공됩니다."),
    ("데이터 말고 다른 혜택도 있나요?",
     "부가서비스 2종이 무료로 제공되며, 로밍 요금이 30% 할인됩니다."),
    ("청구서는 어떻게 받나요?",
     "이메일 또는 우편으로 받으실 수 있습니다. 어느 쪽을 원하시나요?"),
    ("이메일로 해주세요. kim.mj@example.com 입니다.",
     "kim.mj@example.com 으로 청구서를 발송하도록 등록했습니다."),
    ("변경은 언제 적용되나요?",
     "다음 달 1일부터 적용됩니다. 이번 달은 일할 계산으로 청구됩니다."),
    ("지금 쓰던 요금제 약정이 남아 있는데요.",
     "확인해 보겠습니다. 약정 잔여 기간이 4개월 남아 있습니다."),
    ("그럼 위약금이 있나요?",
     "네, 중도 해지 위약금은 32,000원입니다. 다만 상위 요금제로 변경 시에는 면제됩니다."),
    ("면제된다는 거죠?",
     "맞습니다. 프리미엄은 상위 요금제이므로 위약금 32,000원은 면제됩니다."),
    ("가족 결합도 되나요? 회선이 3개 있어요.",
     "3회선 결합 시 회선당 5,500원씩 할인되어 월 16,500원이 절감됩니다."),
    ("좋네요. 그렇게 진행해주세요.",
     "네, 신청이 접수되었습니다. 처리 결과는 문자로 안내드리겠습니다."),
]

def render(turns):
    return "\n".join(f"고객: {u}\n상담원: {a}" for u, a in turns)


print(f"총 {len(TRANSCRIPT)}턴 · 전체 {len(render(TRANSCRIPT)):,}자\n")
print(render(TRANSCRIPT[:3]))

## 준비 3 · 네 가지 방법 구현

**질문** — 어떤 전략들을 비교할까요

### 먼저 "요약으로 대체"가 무엇인지부터

C 와 D 는 대화가 길어지면 **오래된 턴의 원문을 지우고 그 자리에 요약문을 넣습니다.**

```
7턴째 — 아직 그대로
   [1턴][2턴][3턴][4턴][5턴][6턴][7턴]

8턴째 — 오래된 1~5턴을 요약 한 덩어리로 교체
   [1~5턴 요약][6턴][7턴][8턴]
    ↑ 원문 5개가 사라지고 요약문 하나가 그 자리를 대신합니다
```

요약문이 원문보다 훨씬 짧으니 토큰이 줄어듭니다.
대신 **요약에서 빠진 정보는 영영 되찾을 수 없습니다.** 원문을 이미 버렸기 때문입니다.

- **C** 는 그 자리에 **줄글 요약**을 넣습니다
- **D** 는 그 자리에 **`키: 값` 목록**을 넣습니다

### 네 가지 방법

| 전략 | 방식 | 압축 비용 | 예상되는 약점 |
|---|---|---|---|
| **A 전체 유지** | 아무것도 하지 않습니다 (기준선) | 0 | 토큰이 계속 자랍니다 |
| **B 슬라이딩** | 최근 N턴만 남기고 **버립니다** | 0 | 앞턴을 통째로 잊습니다 |
| **C 요약으로 대체** | 오래된 턴 → 줄글 요약 | LLM 호출 | 요약이 정보를 뭉갭니다 |
| **D 핵심만 메모** | 오래된 턴 → `키: 값` 목록 | LLM 호출 | 서술형 맥락이 사라집니다 |

네 전략을 **같은 인터페이스**로 만듭니다 — `build(turns) -> 컨텍스트 문자열`.
인터페이스가 같아야 뒤에서 루프 하나로 공평하게 돌릴 수 있습니다.

### 미리 짚고 갈 것 — 압축은 캐시를 두 번 해칩니다

캐시는 **① 앞부분이 1,024토큰 이상 ② 앞부분이 매번 완전히 동일** 두 조건을 모두 만족해야 걸립니다.
그런데 압축은 **두 조건을 다 건드립니다.**

- **조건 ① 을 깹니다 — 짧아져서 캐시 자체가 걸리지 않습니다**
  A 는 대화가 쌓이며 언젠가 1,024토큰을 넘어 캐시 대상이 됩니다.
  그런데 B·C·D 는 **일부러 짧게 유지**하므로 그 문턱을 영영 넘지 않습니다.
  실제로 이 노트북의 1~6번에서는 **네 전략 모두 캐시가 한 번도 걸리지 않습니다**
  (12턴 시점: A 521 · B 131 · C 362 · D 358 토큰 — 전부 1,024 미달입니다).

- **조건 ② 를 깹니다 — 요약으로 바꾸는 순간 앞부분이 교체됩니다**
  C·D 는 오래된 턴을 요약문으로 갈아끼웁니다. 그 시점에 그동안 쌓인 캐시가 **전부 무효**가 됩니다.
  B 는 매 턴 앞이 잘려나가니 더 나쁩니다.

그래서 **"토큰을 줄였다"가 "돈을 줄였다"로 이어지지 않을 수 있습니다.**
7번에서 캐시가 걸리는 상황을 일부러 만들어 이것을 확인합니다.

### 핵심 설계 — 요약은 매 턴 다시 하지 않습니다

실제 시스템은 이력이 임계치를 넘을 때만 **한 번 요약하고** 그 결과를 재사용합니다.
매 턴 다시 요약하면 두 가지가 망가집니다.

1. 압축기 호출이 턴 수만큼 늘어 **배보다 배꼽**이 커집니다
2. 요약의 요약이 반복되며 정보가 **계단식으로 붕괴**합니다

`MIN_BATCH` 가 그것을 막습니다 — 요약할 분량이 일정량 쌓여야 요약합니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 네 전략을 같은 인터페이스로 만들어 공평하게 비교할 준비를 합니다.
#   · build(turns) -> 컨텍스트 문자열
#   · Summarize 는 만든 요약을 재사용하고 압축기 비용(self.cost)을 누적합니다
#   · MIN_BATCH 로 '매 턴 다시 요약하기'를 막습니다 — 계단식 붕괴 방지
# API 호출: 0회 (정의만)
# ──────────────────────────────────────────────────────────────────────────
KEEP_RECENT = 3        # 최근 몇 턴을 원문으로 유지할지
SUMMARIZE_AFTER = 6     # 이력이 몇 턴을 넘으면 요약을 시작할지
MIN_BATCH = 3  # 한 번에 최소 몇 턴씩 묶어서 요약할지 (매 턴 다시 요약하는 것을 막는다)

SUMMARY_PROMPT = """다음 상담 대화를 4문장 이내로 요약하라.
규칙: 숫자·금액·이메일·최종 결정은 원문 그대로 보존한다.
번복된 내용은 최종 결정만 남긴다. 추측하거나 새 정보를 만들지 않는다.

{text}

요약:"""

FACTS_PROMPT = """다음 상담 대화에서 확정된 사실만 `키: 값` 목록으로 추출하라.
규칙: 숫자·금액·이메일·식별자는 원문 그대로 쓴다.
번복된 내용은 최종 결정만 남긴다. 확정되지 않은 것은 적지 않는다.

{text}

사실 목록:"""


class Strategy:
    """turns 를 받아 모델에 넣을 컨텍스트를 만든다. 압축 비용도 누적 기록한다."""

    def __init__(self, name):
        self.name = name
        self.cost = Usage()     # 압축기가 쓴 토큰 (요약 호출)
        self.digest = ""        # 오래된 턴을 대체할 요약문 (또는 핵심 메모)
        self.done_upto = 0      # 몇 번째 턴까지 요약으로 대체했는지
        self.n_summaries = 0

    def build(self, turns):
        raise NotImplementedError


class FullHistory(Strategy):
    """A) 전체 유지 — 기준선"""
    def build(self, turns):
        return render(turns)


class SlidingWindow(Strategy):
    """B) 슬라이딩 윈도우 — 최근 KEEP_RECENT 턴만. 압축 비용 0, 대신 앞을 잊는다."""
    def __init__(self, name, keep=KEEP_RECENT):
        super().__init__(name); self.keep = keep

    def build(self, turns):
        return render(turns[-self.keep:])


class Summarize(Strategy):
    """C/D) 오래된 턴을 요약문으로 대체하고, 최근 몇 턴만 원문으로 남긴다."""
    def __init__(self, name, prompt, keep=KEEP_RECENT, after=SUMMARIZE_AFTER, min_batch=MIN_BATCH):
        super().__init__(name); self.prompt = prompt
        self.keep = keep; self.after = after; self.min_batch = min_batch

    def build(self, turns):
        # 이력이 임계치를 넘고, 요약할 분량이 min_batch 이상 쌓였을 때만 요약합니다.
        #
        # 매 턴 다시 요약하면 두 가지가 망가집니다.
        #   1) 압축기 호출이 턴 수만큼 늘어 배보다 배꼽이 커집니다
        #   2) 요약의 요약이 반복되며 정보가 계단식으로 붕괴합니다
        if len(turns) > self.after:
            target = len(turns) - self.keep
            if target - self.done_upto >= self.min_batch:
                batch = render(turns[self.done_upto:target])
                prior = f"[이전 요약]\n{self.digest}\n\n" if self.digest else ""
                out, u = generate(self.prompt.format(text=prior + batch))
                self.digest = out
                self.done_upto = target
                self.n_summaries += 1
                self.cost = self.cost + u      # 압축기 비용 누적

        if not self.digest:
            return render(turns)
        return f"[이전 대화 요약]\n{self.digest}\n\n[최근 대화]\n{render(turns[self.done_upto:])}"


def make_strategies():
    return [
        FullHistory("A) 전체 유지"),
        SlidingWindow(f"B) 슬라이딩({KEEP_RECENT}턴)"),
        Summarize("C) 요약으로 대체", SUMMARY_PROMPT),
        Summarize("D) 핵심만 메모", FACTS_PROMPT),
    ]


print(f"최근 {KEEP_RECENT}턴은 원문 유지 · {SUMMARIZE_AFTER}턴 초과 시 요약 시작 · "
      f"한 번에 최소 {MIN_BATCH}턴씩 묶어서 요약")

## 1. 얼마나 줄었나

**질문** — 전략별로 얼마나 줄어들까요

**하는 일** — 같은 대본을 네 전략으로 각각 재생합니다.
매 턴 컨텍스트를 만들고 **실제 API 로 input 토큰을 측정**해 기록합니다.

> 이 노트북에서 가장 오래 걸리는 셀입니다. 12턴 × 4전략 = **48회 호출**입니다 (+ 요약 몇 회).

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 전략별로 토큰이 실제로 얼마나 줄어드는지 측정합니다.
#   · 같은 대본을 네 전략으로 재생하며 턴마다 input 토큰을 실측
#   · ★ 이 노트북에서 가장 오래 걸리는 셀
# API 호출: 12턴 × 4전략 = 48회 (+ 요약 약 4회) · 1~2분
# ──────────────────────────────────────────────────────────────────────────
strategies = make_strategies()
history = {}          # 전략명 -> 턴별 input 토큰
contexts = {}         # 전략명 -> 마지막 컨텍스트 (뒤에서 품질 검증에 사용)

for s in strategies:
    per_turn = []
    for k in range(1, len(TRANSCRIPT) + 1):
        ctx = s.build(TRANSCRIPT[:k])
        per_turn.append(measure(ctx))
        time.sleep(0.2)
    history[s.name] = per_turn
    contexts[s.name] = s.build(TRANSCRIPT)
    print(f"{s.name:18s} 완료 — 마지막 턴 {per_turn[-1]:,} 토큰, 요약 {s.n_summaries}회")

### 턴별 토큰 표

**볼 것** — 각 방법이 턴이 갈수록 어떻게 달라지는지 봅니다.

- **A** 는 숫자가 **계속 커집니다** — 55 → 110 → 165 → … → 521.
  대화가 쌓인 만큼 매번 다 보내기 때문입니다. 01번에서 본 "대화가 부푸는" 그 현상입니다.
- **B** 는 숫자가 **비슷하게 유지됩니다** — 최근 3턴만 보내니 길이가 거의 변하지 않습니다.
- **C·D** 는 커지다가 **한 번 뚝 떨어집니다** — 오래된 턴이 요약문으로 교체된 순간입니다.
  떨어진 지점을 찾으면 언제 요약이 일어났는지 알 수 있습니다.

숫자 하나하나보다 **커지는가 / 유지되는가 / 떨어지는가** 이 세 가지만 보면 됩니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 곡선의 '모양'으로 전략의 성격을 파악합니다.
#   · A 우상향 / B 평평 / C·D 는 요약으로 바꾸는 턴에서 꺾임
# API 호출: 0회 (앞 결과 재사용)
# ──────────────────────────────────────────────────────────────────────────
rows = []
for k in range(len(TRANSCRIPT)):
    rows.append([str(k + 1)] + [f"{history[s.name][k]:,}" for s in strategies])

show_table(
    ["턴"] + [s.name for s in strategies],
    rows,
    align=["right"] + ["right"] * len(strategies),
    title="턴별 input 토큰",
    note="A 는 계속 자라고, B 는 평평하다. C·D 는 요약으로 바꾸는 시점에 한 번 꺾인다.",
)

## 2. 요약이 공짜인가

**질문** — 요약 전략은 공짜일까요?

**아닙니다.** C·D 는 요약을 만들려고 **LLM 을 부릅니다.**
그 비용을 빼고 비교하면 C·D 가 부당하게 유리해 보입니다.

```
총 비용 = 대화 입력 누적  +  압축기가 쓴 토큰(입력+출력)
```

**볼 것** — 압축기 비용을 더했을 때 **순위가 뒤집히는지** 봅니다.
요약이 잦으면 절감액보다 압축 비용이 커집니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 요약 전략이 정말 이득인지, 압축기 비용까지 넣어 재계산합니다.
#   · ★ 압축기 비용을 빼고 비교하면 C·D 가 부당하게 유리해 보입니다
#   · 총 비용 = 대화 입력 누적 + 압축기가 쓴 토큰
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
PRICE = Price(input_per_1k=0.00125, cached_input_per_1k=0.000125, output_per_1k=0.01)

rows, base_total = [], None
for s in strategies:
    conv = sum(history[s.name])
    comp_in, comp_out = s.cost.input_tokens, s.cost.output_tokens
    total = conv + comp_in + comp_out
    base_total = base_total or total
    rows.append([
        s.name, f"{conv:,}",
        f"{comp_in + comp_out:,}" if s.n_summaries else "0",
        f"{total:,}",
        "—" if total == base_total else f"{(1 - total / base_total):+.0%}",
        f"${(conv + comp_in) * PRICE.input_per_1k / 1000 + comp_out * PRICE.output_per_1k / 1000:.5f}",
    ])

show_table(
    ["전략", "대화 입력 누적", "압축기 비용", "총 토큰", "절감률", "비용(USD)"],
    rows,
    align=["left", "right", "right", "right", "right", "right"],
    title=f"{len(TRANSCRIPT)}턴 누적",
    note="압축기 비용을 포함한 총량으로 비교해야 합니다. 요약이 잦으면 배보다 배꼽이 커진다.",
)

## 3. C 와 D 는 뭐가 다른가

**질문** — 두 방법의 실제 차이는 무엇일까요

숫자(토큰 수)만 보면 둘의 차이를 알 수 없습니다. **요약된 결과물을 직접 읽어야** 합니다.

**볼 것 두 가지**

1. **3턴의 번복** — "라이트로 갈까 하다가 프리미엄"이 어떻게 처리됐는지 봅니다.
   최종 결정만 남았으면 성공이고, "라이트"가 남아 있으면 실패입니다.
2. **형식 차이** — C 는 문장이고 D 는 `키: 값` 입니다.
   D 가 더 짧은데도 숫자와 이메일이 더 잘 보존되는지 확인합니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: C 와 D 의 차이를 눈으로 확인합니다. 토큰 수만으로는 알 수 없습니다.
#   · 3턴의 번복이 최종 결정만 남았는지 확인
#   · C 는 문장, D 는 '키: 값' — 어느 쪽이 숫자·이메일을 잘 지키나
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
for s in strategies:
    if not s.digest:
        continue
    print(f"┌─ {s.name} · {s.n_summaries}회 요약함 · {len(s.digest)}자")
    for line in s.digest.splitlines():
        print(f"│ {line}")
    print("└" + "─" * 60 + "\n")

## 4. 줄인 대가는 뭔가

**질문** — 줄이는 데는 성공했습니다. 그런데 답은 맞을까요?

비용만 보면 슬라이딩 윈도우가 압도적입니다. **하지만 답을 못 하면 의미가 없습니다.**

**하는 일** — 12턴이 끝난 시점에 앞쪽에서 정해진 것들을 묻습니다.

| # | 질문 | 정답 | 심어둔 턴 |
|---|---|---|---|
| Q1 | 최종 선택 요금제 | 프리미엄 89,000 | 3턴 (**번복 있음**) |
| Q2 | 청구서 이메일 | kim.mj@example.com | 6턴 |
| Q3 | 위약금 금액과 처리 | 32,000 면제 | 9~10턴 |
| Q4 | 결합 회선 수 | 3회선 | 11턴 (최근) |

**채점 방식** — 정답 문자열이 답변에 살아남았는지 검사합니다(`survival`).
LLM 채점자를 쓰지 않는 이유는 **비용이 0이고 결과가 결정적**이기 때문입니다.

**볼 것** — Q4 는 최근 정보라 **모든 전략이 맞혀야 정상**입니다.
여기서 틀리면 전략 문제가 아니라 구현 버그입니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 줄인 대가로 무엇을 잃었는지 측정합니다.
#   · 12턴 뒤에 앞턴 정보 4개를 묻습니다
#   · 채점은 must_include 문자열 검사(survival) — LLM 채점자 없이 비용 0
#   · Q4(결합)는 최근 정보라 전 전략이 맞혀야 정상
# API 호출: 4질문 × 4전략 = 16회 · 약 30초
# ──────────────────────────────────────────────────────────────────────────
PROBES = [
    ("Q1 요금제",  "고객이 최종적으로 선택한 요금제와 월 요금은?",       ["프리미엄", "89,000"]),
    ("Q2 이메일",  "청구서를 받을 이메일 주소는?",                      ["kim.mj@example.com"]),
    ("Q3 위약금",  "위약금은 얼마이며 부과되는가 면제되는가?",           ["32,000", "면제"]),
    ("Q4 결합",    "가족 결합 회선은 몇 개인가?",                        ["3"]),
]

def norm(s):
    return s.replace(",", "").replace(" ", "").lower()


answers, rows = {}, []
for s in strategies:
    ctx = contexts[s.name]
    got = []
    for label, q, must in PROBES:
        ans, _ = generate(f"{ctx}\n\n질문: {q}\n대화에 근거해 간결히 답하라. "
                          f"근거가 없으면 '모름'이라고 답하라.", max_tokens=120)
        ok = all(norm(m) in norm(ans) for m in must)
        got.append((label, ok, ans))
        time.sleep(0.2)
    answers[s.name] = got
    rows.append([s.name] + ["O" if ok else "X" for _, ok, _ in got]
                + [f"{sum(1 for _, ok, _ in got if ok)}/{len(PROBES)}"])

show_table(
    ["전략"] + [p[0] for p in PROBES] + ["점수"],
    rows,
    align=["left"] + ["center"] * len(PROBES) + ["right"],
    title="앞턴 정보 회상 정확도",
    note="O/X 는 정답 문자열이 답변에 살아남았는지로 판정한다 (survival).",
)

## 5. 어떻게 틀렸나

**질문** — 어떤 방식으로 틀렸을까요?

점수만 보면 무엇이 어떻게 틀렸는지 알 수 없습니다.

**볼 것 — 실패에도 등급이 있습니다.**

- **"모름"이라고 답했다면** → 그나마 안전합니다. 사용자가 다시 물어보면 됩니다
- **틀린 값을 자신 있게 말했다면** → 훨씬 위험합니다. 아무도 오류를 눈치채지 못합니다

압축 전략을 고를 때는 **"틀릴 때 어떻게 틀리는가"** 도 봐야 합니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: '얼마나' 틀렸는지가 아니라 '어떻게' 틀렸는지를 봅니다.
#   · '모름'이라고 답했나, 틀린 값을 지어냈나 — 위험도가 다릅니다
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
fails = [(name, label, ans) for name, got in answers.items()
         for label, ok, ans in got if not ok]

if not fails:
    print("모든 전략이 전부 맞혔다. KEEP_RECENT 를 줄이거나 대화를 늘려 다시 시도해 보라.")
else:
    show_table(
        ["전략", "질문", "실제 답변"],
        [[n, l, a[:60].replace(chr(10), " ") + "…"] for n, l, a in fails],
        align=["left", "left", "left"],
        note="'모름' 이라고 답하면 그나마 안전하다. 잘못된 값을 자신 있게 말하는 게 더 위험하다.",
    )

## 6. 그래서 뭘 쓰나

**질문** — 그래서 무엇을 선택할까요

한쪽만 보면 결론이 뒤집힙니다.

- **비용만** 보면 → B 슬라이딩이 최고입니다
- **품질만** 보면 → A 전체 유지가 최고입니다

**볼 것** — `정답 1개당 토큰` 입니다. 낮을수록 효율적입니다.
다만 **정확도 하한을 먼저 정하고** 그 안에서 비교해야 합니다.
정확도가 낮으면 이 지표는 의미가 없습니다. 틀린 답을 싸게 만드는 것은 가치가 없기 때문입니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 비용과 품질을 한 표에 놓아 실제로 쓸 전략을 고릅니다.
#   · 비용만 보면 B, 품질만 보면 A 가 최고 — 둘을 같이 봐야 합니다
#   · 단, 정확도 하한을 먼저 정하고 그 안에서 토큰을 비교할 것
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
rows = []
for s in strategies:
    total = sum(history[s.name]) + s.cost.input_tokens + s.cost.output_tokens
    score = sum(1 for _, ok, _ in answers[s.name] if ok)
    rows.append([s.name, f"{total:,}",
                 f"{score}/{len(PROBES)}",
                 f"{total / max(score, 1):,.0f}"])

show_table(
    ["전략", "총 토큰", "정확도", "정답 1개당 토큰"],
    rows,
    align=["left", "right", "right", "right"],
    title="비용 × 품질",
    note="'정답 1개당 토큰' 이 낮을수록 효율적이다. 다만 정확도가 낮으면 이 지표는 의미가 없습니다 — "
         "먼저 정확도 하한을 정하고, 그 안에서 토큰을 비교해야 합니다.",
)

## 7. 반전 — 돈은 정말 줄었나

**질문** — 여기까지는 "토큰"을 셌습니다. 그런데 청구서는 토큰이 아니라 **돈**으로 나옵니다.

01번에서 봤듯이 **프롬프트 캐시**가 걸리면 같은 앞부분은 크게 할인됩니다.
그런데 **요약으로 대체는 대화 앞부분을 다른 텍스트로 갈아끼웁니다.**
앞부분이 바뀌면 캐시는 전부 무효가 됩니다.

| 방법 | 앞부분이 | 캐시 |
|---|---|---|
| A 전체 유지 | 고정되고 뒤에만 추가됩니다 | **잘 맞습니다** |
| B 슬라이딩 | 매 턴 잘려나갑니다 | **거의 안 맞습니다** |
| C·D 요약 교체 | 요약하는 순간 교체됩니다 | **요약할 때마다 초기화됩니다** |

> **여기까지의 1~6번에서는 캐시가 아예 걸리지 않았습니다.**
> 대화가 수백 토큰이라 최소 조건(1,024토큰)에 미달했기 때문입니다.
> 그래서 이 절에서는 긴 시스템 지침을 앞에 붙여
> **캐시가 동작하는 상황을 일부러 만든 뒤** 비교합니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 토큰을 줄이면 돈도 줄어드는지 확인합니다. (답: 항상 그렇지는 않다)
#   · 캐시가 걸리도록 1,024토큰 넘는 접두부(POLICY)를 앞에 붙입니다
#   · ★ 전략마다 접두부 태그를 달리해 캐시를 분리 —
#   ·   공유하면 먼저 돌린 쪽이 캐시를 데워줘서 뒤쪽이 유리해집니다
#   · C 가 요약으로 바꾸는 턴에서 적중이 떨어지고 과금이 튀는지 확인
# API 호출: 12턴 × 2전략 = 24회 (+ 요약 약 2회) · 약 1분
# ──────────────────────────────────────────────────────────────────────────
RUN_ID = uuid.uuid4().hex[:8]      # 실행마다 찬 캐시에서 시작 (01번 8절과 같은 이유)

def make_policy(tag):
    """전략마다 접두부를 다르게 만듭니다.

    ★ 이걸 안 하면 실험이 망가집니다.
      A 를 먼저 돌리면 공통 접두부가 데워져서, 뒤에 돌리는 C 가 1턴부터 적중해 버린다.
      '먼저 돌린 쪽이 손해' 라는 순서 효과가 결과를 오염시킵니다.
      전략별로 캐시 공간을 갈라 각자 찬 캐시에서 출발시킵니다.
    """
    return (f"[run={RUN_ID}/{tag}] 당신은 통신사 상담 이력을 분석하는 어시스턴트입니다.\n"
            + "".join(f"규칙{i}. 답변은 대화에 있는 값만 사용하고 추측하지 않으며, "
                      f"금액과 단위를 원문 그대로 표기한다.\n" for i in range(1, 61)))


def cache_run(label, build, tag):
    policy = make_policy(tag)
    rows = []
    for k in range(1, len(TRANSCRIPT) + 1):
        r = responses(f"{policy}\n\n[대화]\n{build(TRANSCRIPT[:k])}\n\n질문: 요약해줘",
                      max_output_tokens=16)
        u = Usage.from_response(r, model=DEPLOYMENT)
        rows.append((k, u.input_tokens, u.cached_tokens, u.billed_input))
        time.sleep(0.4)
    billed = sum(r[3] for r in rows)
    hit = sum(r[2] for r in rows) / max(sum(r[1] for r in rows), 1)
    print(f"{label:18s} 누적 과금입력 {billed:7,d}  평균 적중률 {hit:5.1%}")
    return rows, billed, hit


full_rows, full_billed, full_hit = cache_run("A) 전체 유지", FullHistory("A").build, "full")
sum_rows, sum_billed, sum_hit = cache_run("C) 요약으로 대체", Summarize("C", SUMMARY_PROMPT).build, "fold")

show_table(
    ["턴", "A 입력", "A 캐시", "A 과금", "C 입력", "C 캐시", "C 과금"],
    [[str(a[0]), f"{a[1]:,}", f"{a[2]:,}", f"{a[3]:,}",
      f"{c[1]:,}", f"{c[2]:,}", f"{c[3]:,}"] for a, c in zip(full_rows, sum_rows)],
    align=["right"] * 7,
    foot=["누적", "", "", f"{full_billed:,}", "", "", f"{sum_billed:,}"],
    title="캐시를 포함한 실제 과금 입력 (전략별로 캐시 분리)",
    note=f"A 평균 적중률 {full_hit:.1%} · C 평균 적중률 {sum_hit:.1%}. "
         "C 는 요약으로 바꾸는 턴에서 적중이 떨어집니다 — 접두부가 교체되기 때문입니다.",
)

### 이것이 왜 중요할까요

**"토큰을 줄였다"와 "돈을 줄였다"는 다른 말입니다.**

- 캐시가 잘 맞는 구간에서 요약으로 바꾸면 **줄인 토큰보다 잃은 캐시가 클 수 있습니다**
- 대화가 짧을 때(= 캐시 적중이 이미 높을 때) 요약 교체는 손해입니다
- 요약 교체는 **대화가 충분히 길어져 캐시 이득을 압도할 때** 의미가 생깁니다

02번의 결론과 같습니다 — **"무엇을"보다 "언제"가 중요합니다.**

## 정리

### 전략별 성격

| 전략 | 토큰 | 앞턴 회상 | 압축 비용 | 캐시 | 언제 쓸까요 |
|---|---|---|---|---|---|
| **A 전체 유지** | 최악 | 최고 | 0 | 좋음 | 짧은 대화, 정확도가 최우선일 때 |
| **B 슬라이딩** | 최고 | 최악 | 0 | 나쁨 | 앞 맥락이 필요 없는 단발성 질의 |
| **C 요약으로 대체** | 좋음 | 중간 | 있음 | 요약할 때 초기화 | 긴 대화, 서술형 맥락이 중요할 때 |
| **D 핵심만 메모** | 좋음 | 좋음 | 있음 | 요약할 때 초기화 | **결정과 수치가 중요한 업무 대화** |

### 이 노트북이 준 교훈

- **압축기 비용을 빼고 비교하면 안 됩니다** — C·D 는 LLM 을 부릅니다
- **번복이 있는 대화는 요약이 위험합니다** — "최종 결정만 남겨라"를 프롬프트에 명시해야 합니다
- **구조화(D)가 자연어 요약(C)보다 잘 버팁니다** — 숫자와 이메일이 `키: 값` 형태로 고정되기 때문입니다
- **요약 교체는 캐시를 버립니다** — 토큰 절감과 캐시 손실을 같이 계산해야 합니다

### 남은 숙제

지금은 정답 판정을 `must_include` 문자열 검사로 했습니다. 케이스가 4개라 눈으로 확인했지만,
**`KEEP_RECENT` 와 `SUMMARIZE_AFTER` 를 바꿔가며 수십 조합을 돌리면 눈으로 볼 수 없습니다.**

- 케이스를 **골든셋**으로 분리합니다 (카테고리별: 숫자·부정어·식별자·번복 등)
- `survival` 지표를 함수로 만들고 **최악값과 카테고리별 분해**를 강제합니다
- 그래야 `KEEP_RECENT` 를 2~6 으로 스윕하며 **안전한 하한**을 찾을 수 있습니다